In [ ]:
# ==============================================================================
# Clinical Multiple Linear Regression: Rotator Cuff & Tendinitis Analysis
# Documented and Maintained by: Patricio
# Date: June 2026
# ==============================================================================

# 1. Environment Setup & Requirements
# If packages are missing, please uncomment the following line to install them:
# install.packages(c("tidyverse", "corrplot", "car", "GGally"))

library(tidyverse)
library(corrplot)
library(car)
library(GGally)

set.seed(42) # Ensures reproducibility of clinical distributions

# 2. Dataset Definition & Modeling
# Context: We model Shoulder Pain Intensity (VAS Scale: 0-10) as a function of 
# Patient Age, and specific Visual Analog Scale (VAS) pain indicators for 
# Tendinitis in three key associated muscles: Supraspinatus, Infraspinatus, 
# and Biceps Long Head.
n_patients <- 150

clinical_data <- tibble(
  Patient_ID   = 1:n_patients,
  Age          = round(rnorm(n_patients, mean = 52, sd = 11)),
  
  # Predictor variables: Pain scores on palpation/resisted testing (Scale 0-10)
  Supraspinatus_Tendinitis = round(pmin(pmax(rnorm(n_patients, mean = 6.2, sd = 1.8), 0), 10), 1),
  Infraspinatus_Tendinitis = round(pmin(pmax(rnorm(n_patients, mean = 4.1, sd = 2.0), 0), 10), 1),
  Biceps_LongHead_Tendinitis = round(pmin(pmax(rnorm(n_patients, mean = 4.8, sd = 1.5), 0), 10), 1)
)

# Generate the dependent clinical outcome variable (Overall Shoulder Pain Score)
# incorporating a controlled Gaussian noise component to emulate biological variability.
clinical_data <- clinical_data %>%
  mutate(
    Shoulder_Pain_Score = round(
      pmin(pmax(
        0.05 * Age + 
        0.45 * Supraspinatus_Tendinitis + 
        0.22 * Infraspinatus_Tendinitis + 
        0.18 * Biceps_LongHead_Tendinitis + 
        rnorm(n_patients, mean = 0, sd = 0.75), 
        0
      ), 10), 1
    )
  )

# --- PUBLIC REPOSITORY DOCUMENTATION LINK ---
# For reference on open-access shoulder kinematics and physiotherapy metrics, 
# you can review the SPAR (Shoulder Physiotherapy Activity Recognition) dataset structure:
# URL: https://github.com/dmbee/SPAR-dataset
# ---------------------------------------------

# 3. Exploratory Data Analysis (EDA)
print("=== CLINICAL DATASET SUMMARY ===")
print(summary(clinical_data %>% select(-Patient_ID)))

# 3.1 Distribution and Multivariant Exploration
# This will render a matrix of histograms, scatter plots, and correlation markers.
ggduo_plot <- ggpairs(clinical_data, columns = 2:6, 
                      title = "Clinical Matrix: Rotator Cuff & Muscle Tendinitis Correlates")
print(ggduo_plot)

# 3.2 Core Matrix Evaluation
matrix_cor <- cor(clinical_data %>% select(-Patient_ID))
print("=== CORRELATION MATRIX ===")
print(round(matrix_cor, 3))

# Plotting the correlation heatmap for clinical reporting
corrplot(matrix_cor, method = "color", type = "upper", 
         tl.col = "black", tl.srt = 45, 
         addCoef.col = "black", number.digits = 2,
         mar = c(0,0,2,0), title = "Shoulder Pathology Correlation Heatmap")

# 4. Multiple Linear Regression Formulation
# Equation modeled: 
# Shoulder_Pain_Score = \beta_0 + \beta_1(Age) + \beta_2(Supraspinatus) + \beta_3(Infraspinatus) + \beta_4(Biceps_LongHead) + \epsilon
shoulder_model <- lm(Shoulder_Pain_Score ~ Age + Supraspinatus_Tendinitis + 
                     Infraspinatus_Tendinitis + Biceps_LongHead_Tendinitis, 
                     data = clinical_data)

print("=== MULTIPLE LINEAR REGRESSION ANALYSIS RESULTS ===")
print(summary(shoulder_model))

# 5. Statistical Diagnostics & Verification of Model Assumptions
print("=== MULTICOLLINEARITY DIAGNOSTIC (VIF) ===")
# VIF scores exceeding 5.0 denote severe multicollinearity risks.
print(vif(shoulder_model))

# Global Diagnostic Plot Arrangements
par(mfrow = c(2, 2))
plot(shoulder_model)
par(mfrow = c(1, 1)) # Reset canvas layout

-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.2.1     v readr     2.2.0
v forcats   1.0.1     v stringr   1.6.0
v ggplot2   4.0.3     v tibble    3.3.1
v lubridate 1.9.4     v tidyr     1.3.2
v purrr     1.2.2     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors
corrplot 0.95 loaded
Loading required package: carData

Attaching package: 'car'

The following object is masked from 'package:dplyr':

    recode

The following object is masked from 'package:purrr':

    some

[1] "=== CLINICAL DATASET SUMMARY ==="
      Age        Supraspinatus_Tendinitis Infraspinatus_Tendinitis
 Min.   :19.00   Min.   : 1.300           Min.   :0.000           
 1st Qu.:45.00   1st Qu.: 4.925           1st Qu.:2.800           
 Median :51.50   